# **ETL of Our World in Data: Energy Sources**

## Objectives
* read in Energy Sources data
* clean the data, apply the iso3 country code

## Inputs

* data/raw/electricity-prod-source-stacked/electricity-prod-source-stacked.csv
* data/processed/01b_df_owid_co2_pop_1970_2024.csv

## Outputs

* data/processed/01c_df_owid_co2_pop_src_1970_2024.csv




---

# Set Up directories and Import Necessary Libraries

In [2]:
# System and OS related tasks
import sys
import os

# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

In [3]:
# Get the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid') # sets a white background with grid lines 

# 1.0 OWID Energy Sources Data

In [4]:
data_columns = [
    "country"
    , "country_code"
    , "year"
    , "other_renewables"
    , "bioenergy"
    , "solar"
    , "wind"
    , "hydropower"
    , "nuclear"
    , "oil"
    , "gas"
    , "coal"
    ]
column_dtypes = {
    "country": str
    , "country_code": str
    , "year": int
    , "other_renewables": float
    , "bioenergy": float
    , "solar": float
    , "wind": float
    , "hydropower": float
    , "nuclear": float
    , "oil": float
    , "gas": float
    , "coal": float
}

In [5]:
file_dir_name = "/electricity-prod-source-stacked/electricity-prod-source-stacked.csv"

df_owid_src = pd.read_csv(
    f"{raw_dir}/{file_dir_name}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )

print(f"Total number of rows: {len(df_owid_src)}")

Total number of rows: 6714


In [6]:
df_owid_src

,country,country_code,year,other_renewables,bioenergy,solar,wind,hydropower,nuclear,oil,gas,coal
0,ASEAN (Ember),NaN,2000,16.93,5.87,0.00,0.0,50.45,0.0,65.07,164.26,76.03
1,ASEAN (Ember),NaN,2001,16.40,6.46,0.00,0.0,54.33,0.0,50.99,190.41,86.26
2,ASEAN (Ember),NaN,2002,16.61,6.62,0.00,0.0,53.29,0.0,54.32,208.92,93.43
3,ASEAN (Ember),NaN,2003,15.74,7.45,0.00,0.0,53.21,0.0,53.38,226.51,102.01
4,ASEAN (Ember),NaN,2004,16.57,8.40,0.00,0.0,52.99,0.0,59.50,244.16,115.01
...,...,...,...,...,...,...,...,...,...,...,...,...
6709,Zimbabwe,ZWE,2019,0.00,0.19,0.02,0.0,4.17,0.0,0.05,0.00,4.05
6710,Zimbabwe,ZWE,2020,0.00,0.10,0.02,0.0,3.81,0.0,0.04,0.00,2.74
6711,Zimbabwe,ZWE,2021,0.00,0.11,0.02,0.0,5.93,0.0,0.00,0.00,2.51
6712,Zimbabwe,ZWE,2022,0.00,0.11,0.03,0.0,5.88,0.0,0.00,0.00,2.92


In [7]:
# look at the unique country codes
unique_codes = df_owid_src['country_code'].dropna().unique()
print(f"Total unique country codes: {len(unique_codes)}")

Total unique country codes: 222


In [9]:
# add the iso country code 
import country_converter as coco
cc = coco.CountryConverter()

df_owid_src["country_code_iso3"] = cc.pandas_convert(
    series=df_owid_src['country_code'],
    to='ISO3'
)


nan not found in ISO3
OWID_AFR not found in regex
OWID_ASI not found in regex
OWID_EUR not found in regex
OWID_EU27 not found in regex
OWID_HIC not found in regex
OWID_KOS not found in regex
OWID_LIC not found in regex
OWID_LMC not found in regex
OWID_NAM not found in regex
OWID_OCE not found in regex
OWID_SAM not found in regex
OWID_UMC not found in regex
OWID_WRL not found in regex


📌 We already know about "OWID_KOS" and "OWID_WRL" and so these other `country_code` values look like aggregations rows which we do not want in our main dataset:

In [10]:
country_code_mapping = {
    "OWID_AFR"              # Africa
    , "OWID_ASI"            # Asia 
    , "OWID_EUR"            # Europe  
    , "OWID_HIC"            # High Income countries  
    , "OWID_LIC"            # Low Income countries   
    , "OWID_LMC"            # Low Middle Income countries   
    , "OWID_NAM"            # North America
    , "OWID_OCE"            # Oceania  
    , "OWID_SAM"            # South America  
    , "OWID_UMC"            # Upper Middle Income countries   
    , "OWID_WRL"            # WORLD
    , "UN_AFR"              # Africa UN Version
    , "UN_ASI"              # Asia UN Version
    , "UN_EUR"              # Europe UN Version
    , "UN_LAC"              # Latin America & Caribbean UN Version 
    , "UN_NAM"              # North America UN Version 
    , "UN_OCE"              # Oceania UN Version 
}

📌 Check the missing values in `country_code` and the data looks like aggregations too.

In [11]:
# find the missing values in country_code
nan_rows =df_owid_src[df_owid_src['country_code'].isna()]

nan_rows['country'].value_counts()

country
Africa (EI)                            40
Asia Pacific (EI)                      40
CIS (EI)                               40
Europe (EI)                            40
Middle East (EI)                       40
Non-OECD (EI)                          40
North America (EI)                     40
OECD (EI)                              40
South and Central America (EI)         40
EU (Ember)                             36
ASEAN (Ember)                          25
Africa (Ember)                         25
Asia (Ember)                           25
Europe (Ember)                         25
G20 (Ember)                            25
G7 (Ember)                             25
Latin America and Caribbean (Ember)    25
Middle East (Ember)                    25
North America (Ember)                  25
OECD (Ember)                           25
Oceania (Ember)                        25
Name: count, dtype: int64

In [109]:
nan_rows.head(10)

,country,country_code,year,pop_actual,pop_projected,country_code_iso3
906,Americas (UN),NaN,1950,335791496.0,NaN,not found
907,Americas (UN),NaN,1951,342809279.0,NaN,not found
908,Americas (UN),NaN,1952,350086867.0,NaN,not found
909,Americas (UN),NaN,1953,357599609.0,NaN,not found
910,Americas (UN),NaN,1954,365377091.0,NaN,not found
911,Americas (UN),NaN,1955,373407276.0,NaN,not found
912,Americas (UN),NaN,1956,381692163.0,NaN,not found
913,Americas (UN),NaN,1957,390249080.0,NaN,not found
914,Americas (UN),NaN,1958,398992351.0,NaN,not found
915,Americas (UN),NaN,1959,408077039.0,NaN,not found


## 2.1 Country Code "OWID_KOS"

📌 The `country_code` column in the OWID data with the value `OWID_KOS` refers to the country Kosovo. 

Kosovo is not yet a member of the United Nations and as such the Country_Converter package [on Kosovo](https://github.com/vincentarelbundock/countrycode/issues/305) does know what ISO3 code to assign to it.

Many commercial datasets and the World's bank has provisionally used `XKX` for Kosovo. We will manually use that as the country_code instead.

In [12]:
kos_rows = df_owid_src[df_owid_src['country_code'] == 'OWID_KOS']

kos_rows.head(5)

,country,country_code,year,other_renewables,bioenergy,solar,wind,hydropower,nuclear,oil,gas,coal,country_code_iso3
3299,Kosovo,OWID_KOS,2000,0.0,0.0,0.0,0.0,0.05,0.0,0.02,0.0,2.89,not found
3300,Kosovo,OWID_KOS,2001,0.0,0.0,0.0,0.0,0.05,0.0,0.02,0.0,3.66,not found
3301,Kosovo,OWID_KOS,2002,0.0,0.0,0.0,0.0,0.05,0.0,0.02,0.0,3.64,not found
3302,Kosovo,OWID_KOS,2003,0.0,0.0,0.0,0.0,0.05,0.0,0.03,0.0,3.55,not found
3303,Kosovo,OWID_KOS,2004,0.0,0.0,0.0,0.0,0.11,0.0,0.03,0.0,3.94,not found


In [13]:
# find the rows with country_code = "OWID_KOS" and assign the value "XKX" to the country_code_iso3

df_owid_src["country_code_iso3"] = df_owid_src["country_code"].replace("OWID_KOS", "XKX")


kos_rows = df_owid_src[df_owid_src['country_code'] == 'OWID_KOS']
kos_rows.head(5)

,country,country_code,year,other_renewables,bioenergy,solar,wind,hydropower,nuclear,oil,gas,coal,country_code_iso3
3299,Kosovo,OWID_KOS,2000,0.0,0.0,0.0,0.0,0.05,0.0,0.02,0.0,2.89,XKX
3300,Kosovo,OWID_KOS,2001,0.0,0.0,0.0,0.0,0.05,0.0,0.02,0.0,3.66,XKX
3301,Kosovo,OWID_KOS,2002,0.0,0.0,0.0,0.0,0.05,0.0,0.02,0.0,3.64,XKX
3302,Kosovo,OWID_KOS,2003,0.0,0.0,0.0,0.0,0.05,0.0,0.03,0.0,3.55,XKX
3303,Kosovo,OWID_KOS,2004,0.0,0.0,0.0,0.0,0.11,0.0,0.03,0.0,3.94,XKX


## 2.2 Aggregations Country Code values and nan

📌 The `country_code` column in the OWID population data has the following values that refes to the aggregation of populations at different granularity. Any rows with nan in the same country is also an aggregations.

In [14]:
country_code_mapping

{'OWID_AFR',
 'OWID_ASI',
 'OWID_EUR',
 'OWID_HIC',
 'OWID_LIC',
 'OWID_LMC',
 'OWID_NAM',
 'OWID_OCE',
 'OWID_SAM',
 'OWID_UMC',
 'OWID_WRL',
 'UN_AFR',
 'UN_ASI',
 'UN_EUR',
 'UN_LAC',
 'UN_NAM',
 'UN_OCE'}

In [15]:
# create a  query mask for identifying rows with with aggregation data or is nan
exclude_mask = (df_owid_src["country_code"].isin(country_code_mapping) ) | (df_owid_src["country_code"].isna())

### 2.2.1 Get and Check the World aggregation data

In [16]:
# get a copy of the df_owid_co2 by including all the unwanted rows
df_owid_src_world = df_owid_src[exclude_mask].copy()

In [ ]:
num_aggre_rows = (df_owid_src["country_code"].isin(country_code_mapping)).sum()
num_nan_rows = df_owid_src["country_code"].isna().sum()

print(f"Total rows in df_owid_src: {len(df_owid_src)}")
print(f"OWID_WRL rows: {num_aggre_rows}")
print(f"NaN rows: {num_nan_rows}")
print(f"Total rows to be excluded: {num_aggre_rows + num_nan_rows}")
print(f"Country Specific rows left: {len(df_owid_src) - num_aggre_rows - num_nan_rows}")

Total ros in df_owid_src: 6714
OWID_WRL rows: 307
NaN rows: 671
Total rows to be excluded: 978
Country Specific rows left: 5736


📌 Export to csv

In [ ]:
#df_owid_src_world.to_csv(f'{processed_dir}/01c_df_owid_src_world.csv', index=False)
#print(f"Exported: {processed_dir}/01c_df_owid_src_world.csv")

Exported: ../data/processed/01b_df_owid_pop_world.csv


### 2.2.2 Get and Check the country-year dataset

In [20]:
# get a copy of the df_owid_co2 by excluding all the unwanted rows
df_owid_src_country = df_owid_src[~exclude_mask].copy()

In [21]:
check_kosovo = df_owid_src_country[df_owid_src_country["country"] == "Kosovo"]

check_kosovo.head(2)

,country,country_code,year,other_renewables,bioenergy,solar,wind,hydropower,nuclear,oil,gas,coal,country_code_iso3
3299,Kosovo,OWID_KOS,2000,0.0,0.0,0.0,0.0,0.05,0.0,0.02,0.0,2.89,XKX
3300,Kosovo,OWID_KOS,2001,0.0,0.0,0.0,0.0,0.05,0.0,0.02,0.0,3.66,XKX


In [22]:
df_owid_src_country["year"].describe()

count    5736.000000
mean     2009.436192
std         9.190919
min      1985.000000
25%      2003.000000
50%      2010.000000
75%      2017.000000
max      2025.000000
Name: year, dtype: float64

In [38]:
df_owid_src_country_2000_2024 = df_owid_src_country[
    (df_owid_src_country["year"] >= 2000) &
    (df_owid_src_country["year"] <= 2024)
]

📌 Verify by country whether there are missing years.


In [39]:
#check for duplcates country-year rows
country_year_counts = df_owid_src_country_2000_2024.groupby(
    ["country", "year"]
).size()

print(f"Number of duplicated country-year: {len(country_year_counts[country_year_counts > 1])}")

Number of duplicated country-year: 0


# 2.4 Export Energy Sources data frame to csv

In [40]:
df_owid_src_country_2000_2024.to_csv(f'{processed_dir}/01c_df_owid_src_country_2000_2024.csv', index=False)
print(f"Exported: {processed_dir}/01c_df_owid_src_country_2000_2024.csv")

Exported: ../data/processed/01c_df_owid_src_country_2000_2024.csv


---